# `asyncio`
- **no** multiple processes, **neither** multiple threads
- runs **seemingly** in parallel, but **_en fait_** in **very rapid alternation**

In [1]:
import time
import asyncio

In [2]:
for k in range(10, 1, -1):
    print(k)

10
9
8
7
6
5
4
3
2


In [3]:
any([False, False, False])

False

In [4]:
any([False, False, True])

True

In [5]:
all([False, False, False])

False

In [6]:
all([False, False, True])

False

I don't quite like the `is_prime` function written by Sebastiaan, so I rewrote similar ones. Let's test their efficiency.

In [7]:
def is_prime1(k):
    return not any((k % i) == 0 for i in range(2, k))

def is_prime2(k):
    for i in range(2, k):
        if (k % i) == 0:
            return False
    return True

In [8]:
%%timeit
is_prime1(int(1e10))

1.67 µs ± 131 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [9]:
%%timeit
is_prime2(int(1e10))

797 ns ± 34.2 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [10]:
is_prime = is_prime2

In [11]:
print(None)

None


In [12]:
def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

def main():
    biggest_prime_le(int(1e7))
    biggest_prime_le(100_000)
    biggest_prime_le(10_000)
    biggest_prime_le(1_000)


main()

(10000000) Finding...
(10000000) Found 9999991
(  100000) Finding...
(  100000) Found 99991
(   10000) Finding...
(   10000) Found 9973
(    1000) Finding...
(    1000) Found 997


#### Sth interesting
- The biggest prime $\le$ `1e7` is `9999991`
- The biggest prime $\le$ `1e5` is `99991`

Coincidence?

```
(1000000000) Finding...
(1000000000) Found 999999937
```

## Little by little
Let's carry the code little by little towards asynchronous code like Sebastiaan did in the video

### 1st: adding `async` to `main()`

In [13]:
def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    biggest_prime_le(int(1e7))
    biggest_prime_le(100_000)
    biggest_prime_le(10_000)
    biggest_prime_le(1_000)


main()

<coroutine object main at 0x7f5fc05f90e0>

Like Sebastiaan said in the video, `async` functions are <b>not</b> supposed to be called like this.
<br>
Instead, they are either to be <b><code>await</code></b>ed or to be put inside an <code><b>event_loop</b></code>, as we will briefly see.


In [16]:
def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    biggest_prime_le(int(1e7))
    biggest_prime_le(100_000)
    biggest_prime_le(10_000)
    biggest_prime_le(1_000)

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

/home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages/traitlets/config/configurable.py:118: RuntimeWarning: coroutine 'main' was never awaited
  return  [c.__name__ for c in reversed(cls.__mro__) if


RuntimeError: This event loop is already running

In [19]:
pip show tornado

Name: tornado
Version: 6.0.4
Summary: Tornado is a Python web framework and asynchronous networking library, originally developed at FriendFeed.
Home-page: http://www.tornadoweb.org/
Author: Facebook
Author-email: python-tornado@googlegroups.com
License: http://www.apache.org/licenses/LICENSE-2.0
Location: /home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages
Requires: 
Required-by: terminado, notebook, jupyter-client, ipykernel
Note: you may need to restart the kernel to use updated packages.


### What happened? Why the behaviour is diff from in the video?
The story was sth related to package versions and `jupyter`. In brief summary, as far as I understand,
> `jupyter` itself runs an event loop and does not allow another event loop to be run in any of its cells

Two solutions:
- Use `nest_asyncio` like below
- Downgrade `tornado` to `tornado==4.5.3`

**cf.**
- [https://stackoverflow.com/questions/46827007/runtimeerror-this-event-loop-is-already-running-in-python](https://stackoverflow.com/questions/46827007/runtimeerror-this-event-loop-is-already-running-in-python)
- [https://medium.com/@vyshali.enukonda/how-to-get-around-runtimeerror-this-event-loop-is-already-running-3f26f67e762e](https://medium.com/@vyshali.enukonda/how-to-get-around-runtimeerror-this-event-loop-is-already-running-3f26f67e762e)
- [https://stackoverflow.com/questions/53248431/asyncio-runtimeerror-this-event-loop-is-already-running/53525009](https://stackoverflow.com/questions/53248431/asyncio-runtimeerror-this-event-loop-is-already-running/53525009)

In [20]:
!pip install nest_asyncio

You should consider upgrading via the '/home/phunc20/.virtualenvs/async-by-sebastiaan/bin/python -m pip install --upgrade pip' command.


In [21]:
import nest_asyncio
nest_asyncio.apply()

In [22]:
def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    biggest_prime_le(int(1e7))
    biggest_prime_le(100_000)
    biggest_prime_le(10_000)
    biggest_prime_le(1_000)

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

(10000000) Finding...
(10000000) Found 9999991
(  100000) Finding...
(  100000) Found 99991
(   10000) Finding...
(   10000) Found 9973
(    1000) Finding...
(    1000) Found 997


Now, the code runs, but **not** asynchronously.

Let's try to add `async` keyword before the function `biggest_prime_le()`.

In [23]:
async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    biggest_prime_le(int(1e7))
    biggest_prime_le(100_000)
    biggest_prime_le(10_000)
    biggest_prime_le(1_000)

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

/home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages/ipykernel_launcher.py:11: RuntimeWarning: coroutine 'biggest_prime_le' was never awaited
  # This is added back by InteractiveShellApp.init_path()
/home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages/ipykernel_launcher.py:12: RuntimeWarning: coroutine 'biggest_prime_le' was never awaited
  if sys.path[0] == '':
/home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages/ipykernel_launcher.py:13: RuntimeWarning: coroutine 'biggest_prime_le' was never awaited
  del sys.path[0]
/home/phunc20/.virtualenvs/async-by-sebastiaan/lib/python3.7/site-packages/ipykernel_launcher.py:14: RuntimeWarning: coroutine 'biggest_prime_le' was never awaited
  


To fix this, let's `await` those `biggest_prime_le()` inside `main()`.

In [24]:
async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    await biggest_prime_le(int(1e7))
    await biggest_prime_le(100_000)
    await biggest_prime_le(10_000)
    await biggest_prime_le(1_000)

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

(10000000) Finding...
(10000000) Found 9999991
(  100000) Finding...
(  100000) Found 99991
(   10000) Finding...
(   10000) Found 9973
(    1000) Finding...
(    1000) Found 997


now, let's introduce a new term: **`asyncio.wait()`**

In [26]:
async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        time.sleep(0.01)
    return None

async def main():
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

(    1000) Finding...
(    1000) Found 997
(  100000) Finding...
(  100000) Found 99991
(10000000) Finding...
(10000000) Found 9999991
(   10000) Finding...
(   10000) Found 9973


Now, this is <b>slightly asynchronous</b>: The order of <code>print</code>s is now different.

### Conciously adding a point of suspension in our code
<b><code>asyncio.sleep()</code></b> instead of <code>time.sleep()</code>

In [27]:
async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        await asyncio.sleep(0.01)
        #time.sleep(0.01)
    return None

async def main():
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

(    1000) Finding...
(   10000) Finding...
(10000000) Finding...
(  100000) Finding...
(    1000) Found 997
(10000000) Found 9999991
(  100000) Found 99991
(   10000) Found 9973


**Note** how those `Finding...` are all printed out first this time.

### Let's time it

In [36]:
%%time

print("Asynchronous code (i.e. using async.sleep())")

async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        await asyncio.sleep(0.01)
        #time.sleep(0.01)
    return None

async def main():
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

Asynchronous code (i.e. using async.sleep())
(   10000) Finding...
(10000000) Finding...
(  100000) Finding...
(    1000) Finding...
(    1000) Found 997
(10000000) Found 9999991
(  100000) Found 99991
(   10000) Found 9973
CPU times: user 716 ms, sys: 9.87 ms, total: 726 ms
Wall time: 986 ms


In [40]:
%%time

print("Synchronous code (i.e. using time.sleep())")

async def biggest_prime_le(n):
    print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            print(f"({n:8d}) Found {k}")
            return k
        #await asyncio.sleep(0.01)
        time.sleep(0.01)
    return None

async def main():
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

Synchronous code (i.e. using time.sleep())
(  100000) Finding...
(  100000) Found 99991
(10000000) Finding...
(10000000) Found 9999991
(    1000) Finding...
(    1000) Found 997
(   10000) Finding...
(   10000) Found 9973
CPU times: user 781 ms, sys: 6.54 ms, total: 788 ms
Wall time: 1.27 s


In [49]:
print("Asynchronous code (i.e. using async.sleep())")

async def biggest_prime_le(n):
    #print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            #print(f"({n:8d}) Found {k}")
            return k
        await asyncio.sleep(0.01)
        #time.sleep(0.01)
    return None

async def main():
    t0 = time.time()
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])
    t1 = time.time()
    print(f"Took {1000*(t1-t0)} ms")

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

Asynchronous code (i.e. using async.sleep())
Took 971.7938899993896 ms


In [54]:
print("Synchronous code (i.e. using time.sleep())")

async def biggest_prime_le(n):
    #print(f"({n:8d}) Finding...")
    for k in range(n, 1, -1):
        if is_prime(k):
            #print(f"({n:8d}) Found {k}")
            return k
        #await asyncio.sleep(0.01)
        time.sleep(0.01)
    return None

async def main():
    t0 = time.time()
    await asyncio.wait([
          biggest_prime_le(int(1e7)),
          biggest_prime_le(100_000),
          biggest_prime_le(10_000),
          biggest_prime_le(1_000),
    ])
    t1 = time.time()
    print(f"Took {1000*(t1-t0)} ms")

loop = asyncio.get_event_loop()
loop.run_until_complete(main())
## Normally we need the next line to close things up, but inside jupyter notebook: No.
#loop.close()

Synchronous code (i.e. using time.sleep())
Took 1202.3682594299316 ms
